# Sokhan: fine-tune Whisper for Persian on a free GPU

**Runtime: GPU (T4).** Runtime → Change runtime type → T4 GPU.

این نوتبوک کل زنجیره‌ی fine-tune فارسی را روی **GPU رایگان** اجرا می‌کند:

1. گرفتن سورس پروژه و نصب وابستگی‌ها
2. دانلود داده‌ی واقعی Common Voice (فارسی، CC-0)
3. آماده‌سازی manifest و آموزش LoRA روی `whisper-small`
4. ادغام adapter و تبدیل به فرمت CTranslate2 (موتور خود Sokhan)
5. تست واقعی با faster-whisper + دانلود / آپلود به Hugging Face Hub

پایپ‌لاین قبلاً روی CPU با `training/smoke_test.py` تایید شده است؛ اینجا همان کد روی GPU واقعی اجرا می‌شود. اعداد WER فقط **اندازه‌گیری‌شده** گزارش می‌شوند — هرگز حدس زده نمی‌شوند.

مستندات کامل: [training/README.md](https://github.com/Abolfazlrwm/sokhan/blob/main/training/README.md)

In [ ]:
#@title 1. GPU check
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0), "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB VRAM")

In [ ]:
#@title 2. Clone Sokhan and install training deps
!git clone https://github.com/Abolfazlrwm/sokhan.git
%cd sokhan
# torch is already on Colab; this installs the rest (transformers, peft,
# datasets, soundfile, jiwer, ctranslate2 for the conversion step)
!pip install -q -r training/requirements.txt ctranslate2
# faster-whisper's decoder needs PyAV < 14 (verified: newer PyAV removed an API it uses)
!pip install -q "av<14" faster-whisper

In [ ]:
#@title 3. Sanity-check the pipeline before spending GPU time (~1 min)
!python training/smoke_test.py

In [ ]:
#@title 4. Hugging Face token (free account is enough)
# 1) حساب رایگان بسازید: https://huggingface.co/join
# 2) روی دیتاست گیت‌شده کلیک کنید و شرایط را accept کنید:
#    https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0
# 3) توکن بسازید: https://huggingface.co/settings/tokens
import getpass
HF_TOKEN = getpass.getpass("HF token (input hidden): ")

In [ ]:
#@title 5. Download real Common Voice (fa) and build the manifest
# Same filter rules as training/prepare_common_voice.py: up-votes >= 2,
# no down-votes, 1-30 s, transcript >= 4 chars. Audio is written to disk so
# the repo's training scripts work unchanged.
MAX_HOURS = 10  #@param {type:"number"}

import json
from pathlib import Path

import soundfile as sf
from datasets import load_dataset

out_dir = Path("training/data/cv-fa/clips")
out_dir.mkdir(parents=True, exist_ok=True)
manifest_path = Path("training/data/cv-fa/manifest.jsonl")

ds = load_dataset(
    "mozilla-foundation/common_voice_17_0",
    "fa",
    split="train",
    streaming=True,
    token=HF_TOKEN,
)

kept = skipped = 0
total_seconds = 0.0
with manifest_path.open("w", encoding="utf-8") as sink:
    for row in ds:
        sentence = (row.get("sentence") or "").strip()
        if row.get("down_votes", 0) > 0 or row.get("up_votes", 0) < 2:
            skipped += 1
            continue
        if len(sentence) < 4:
            skipped += 1
            continue
        audio = row["audio"]
        duration = len(audio["array"]) / audio["sampling_rate"]
        if not (1.0 <= duration <= 30.0):
            skipped += 1
            continue
        if total_seconds + duration > MAX_HOURS * 3600:
            break
        clip = out_dir / f"clip_{kept:06d}.wav"
        sf.write(str(clip), audio["array"], audio["sampling_rate"])
        sink.write(json.dumps(
            {"audio": str(clip), "text": sentence, "duration_seconds": round(duration, 3)},
            ensure_ascii=False,
        ) + "\n")
        kept += 1
        total_seconds += duration

print(f"kept: {kept} clips ({total_seconds / 3600:.2f} h), skipped: {skipped}")
print(f"manifest: {manifest_path}")

In [ ]:
#@title 6. LoRA fine-tune whisper-small on the free T4
# First run: keep --max-samples small (e.g. 3000) to see the loop work,
# then raise it. Epoch-wise eval WER uses Sokhan's own Persian normalization,
# so numbers are comparable with benchmarks/benchmark_wer.py.
EPOCHS = 3  #@param {type:"number"}
MAX_SAMPLES = 0  #@param {type:"integer"}

!python training/train_whisper_lora.py \
    --manifest training/data/cv-fa/manifest.jsonl \
    --output training/runs/whisper-small-fa-lora \
    --epochs {EPOCHS} \
    --max-samples {MAX_SAMPLES}

In [ ]:
#@title 7. Convert to CTranslate2 (the format faster-whisper serves)
!ct2-transformers-converter \
    --model training/runs/whisper-small-fa-lora/merged \
    --output_dir whisper-small-fa-ct2 \
    --quantization float16
!du -sh whisper-small-fa-ct2

In [ ]:
#@title 8. Real serving test with faster-whisper (the actual Sokhan engine)
import json
from faster_whisper import WhisperModel

model = WhisperModel("whisper-small-fa-ct2", device="cuda", compute_type="float16")

# transcribe one held-out clip and score it against its reference
with open("training/data/cv-fa/manifest.jsonl", encoding="utf-8") as handle:
    last = json.loads(handle.readlines()[-1])

segments, info = model.transcribe(last["audio"], language="fa", beam_size=5)
hypothesis = " ".join(s.text for s in segments)
print("reference :", last["text"])
print("hypothesis:", hypothesis)

In [ ]:
#@title 9. Save the model — pick one
SAVE_MODE = "zip"  #@param ["zip", "hf_hub"]

if SAVE_MODE == "zip":
    from google.colab import files
    !zip -r whisper-small-fa-ct2.zip whisper-small-fa-ct2
    files.download("whisper-small-fa-ct2.zip")
else:
    from huggingface_hub import login, create_repo, upload_folder
    login(token=HF_TOKEN)
    repo_id = input("repo id, e.g. yourname/whisper-small-fa-sokhan: ")
    create_repo(repo_id, private=False, exist_ok=True)
    upload_folder(repo_id=repo_id, folder_path="whisper-small-fa-ct2")
    print("uploaded to:", f"https://huggingface.co/{repo_id}")

## بعدش چی؟ (Next steps)

- **سرو کردن:** فولدر `whisper-small-fa-ct2` را در سرور خودت بگذار و `WHISPER_MODEL_SIZE` را به مسیرش بده (deploy/docker-compose.yml).
- **بنچمارک واقعی:** کپی این فولدر را کنار `benchmarks/benchmark_wer.py` بگذار و اجرا کن:
  `python benchmarks/benchmark_wer.py --models whisper-small-fa-ct2` — بعد جدول اندازه‌گیری‌شده را با یک PR کنار baseline در `benchmarks/` اضافه کن.
- **ادامه‌ی تمرین:** برای ران کامل، `MAX_HOURS` و `MAX_SAMPLES` را بالا ببرید؛ کووتای رایگان Kaggle (۳۰ ساعت/هفته) برای ران‌های طولانی‌تر منظم‌تر است — همین نوتبوک در Kaggle هم کار می‌کند.
- عدد WER را فقط بعد از اجرای واقعی گزارش کنید؛ قانون پروژه: **اندازه‌گیری‌شده یا هیچ**.